# ДЗ7. Часові ряди та напівструктуровані дані
**Звірковська Ірина**

Я використала реальні NYC TLC Yellow Taxi за березень 2024 року та синтетичний журнал із 50 000 подій. Вибірка містить 500 000 поїздок з усього місяця до очищення.

Джерело: [NYC TLC](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). Запуск: **Runtime → Restart session and run all**. Усі навчальні таблиці та індекси перевідтворюються автоматично.

**Перевірка виконання:** збережені outputs отримано локально на macOS, Python 3.12.14, PostgreSQL 16.2. Два послідовні запуски всіх кодових комірок із новим ядром пройшли без помилок, зокрема повторне створення об’єктів у тій самій базі.

## 1. Setup

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow requests
from pathlib import Path
from urllib.request import urlretrieve
import json
import platform
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text
from sqlalchemy.engine import make_url
from IPython.display import display, Markdown

DATA_DIR = Path('/content') if Path('/content').exists() else Path.cwd() / 'hw7_data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(make_url(pg.get_uri()).set(drivername='postgresql+psycopg2'), future=True)
with engine.begin() as conn:
    conn.execute(text("SET TIME ZONE 'UTC'"))
    print(conn.execute(text('SELECT version()')).scalar_one())
print('Python:', platform.python_version(), '| OS:', platform.system())

# У цих навчальних SQL-блоках крапки з комою розділяють окремі запити.
def run_sql(sql):
    with engine.begin() as conn:
        for statement in sql.split(';'):
            if not statement.strip():
                continue
            result = conn.execute(text(statement))
            if result.returns_rows:
                display(pd.DataFrame(result.fetchall(), columns=result.keys()))

plans = {}
def explain(label, query):
    with engine.connect() as conn:
        lines = conn.execute(text('EXPLAIN (ANALYZE, BUFFERS) ' + query)).scalars().all()
    plans[label] = '\n'.join(lines)
    print(label + '\n' + plans[label])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 88.5 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit
Python: 3.12.12 | OS: Linux


## 2. NYC Taxi: вибірка та очищення
Я спочатку обмежила pickup-час березнем, щоб випадкові аномальні дати не потрапили до календаря. Далі відібрала 500 000 рядків із фіксованим `random_state=42`. Це вибіркові, а не повні місячні підрахунки попиту.

In [2]:
URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet'
parquet_path = DATA_DIR / 'yellow_tripdata_2024-03.parquet'
if not parquet_path.exists():
    urlretrieve(URL, parquet_path)
columns = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID',
           'DOLocationID', 'trip_distance', 'fare_amount', 'passenger_count']
full = pd.read_parquet(parquet_path, columns=columns)
print('Full dataset:', full.shape)
in_month = full['tpep_pickup_datetime'].ge('2024-03-01') & full['tpep_pickup_datetime'].lt('2024-04-01')
print('Outside March:', int((~in_month).sum()))
df = full.loc[in_month].sample(n=500_000, random_state=42).sort_values('tpep_pickup_datetime').reset_index(drop=True)
del full
print('Sample:', df.shape)
print('Pickup range:', df['tpep_pickup_datetime'].min(), '—', df['tpep_pickup_datetime'].max())
assert len(df) == 500_000
df = df.loc[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)
            & (df['trip_distance'] > 0) & (df['trip_distance'] < 100)
            & df['PULocationID'].notna()].copy()
df['PULocationID'] = df['PULocationID'].astype('int64')
print('After cleaning:', df.shape)
df.to_sql('nyc_taxi_raw', engine, if_exists='replace', index=False, chunksize=10_000)
run_sql('SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw;')

Full dataset: (3582628, 7)
Outside March: 23
Sample: (500000, 7)
Pickup range: 2024-03-01 00:00:03 — 2024-03-31 23:59:56
After cleaning: (480094, 7)


,n_rows
0,480094


## 3. Daily aggregation і calendar-spine
TLC timestamps — локальний час Нью-Йорка, тому день обчислюю без зміни часової зони. Для event-log далі використовую `TIMESTAMPTZ`. Для кожної наявної у вибірці зони створюю всі 31 день. Дні без відібраних поїздок отримують `trip_count = 0`, `total_distance = 0`, але `avg_fare = NULL`: середня ціна без поїздок невідома. Це забезпечує календарний зміст lag-ознак.

In [3]:
run_sql("""
DROP VIEW IF EXISTS v_t7_hw_taxi_features;
DROP TABLE IF EXISTS t7_hw_taxi_daily;
CREATE TABLE t7_hw_taxi_daily AS
WITH daily AS (
 SELECT "PULocationID"::INTEGER AS pickup_zone_id,
        tpep_pickup_datetime::DATE AS ts_day,
        COUNT(*)::INTEGER AS trip_count,
        AVG(fare_amount)::NUMERIC(10,2) AS avg_fare,
        SUM(trip_distance)::NUMERIC(12,2) AS total_distance
 FROM nyc_taxi_raw GROUP BY 1,2
), zones AS (SELECT DISTINCT pickup_zone_id FROM daily),
calendar AS (
 SELECT d::DATE AS ts_day FROM generate_series(TIMESTAMP '2024-03-01', TIMESTAMP '2024-03-31', INTERVAL '1 day') AS d
)
SELECT z.pickup_zone_id, c.ts_day, COALESCE(d.trip_count,0)::INTEGER AS trip_count,
       d.avg_fare, COALESCE(d.total_distance,0)::NUMERIC(12,2) AS total_distance
FROM zones z CROSS JOIN calendar c LEFT JOIN daily d USING (pickup_zone_id,ts_day);
ALTER TABLE t7_hw_taxi_daily ADD PRIMARY KEY (pickup_zone_id,ts_day);
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT pickup_zone_id) AS n_zones,
       MIN(ts_day) AS first_day, MAX(ts_day) AS last_day,
       COUNT(*) FILTER (WHERE trip_count=0) AS zero_trip_days FROM t7_hw_taxi_daily;
SELECT pickup_zone_id, COUNT(*) AS n_days FROM t7_hw_taxi_daily GROUP BY 1 ORDER BY 1 LIMIT 10;
""")

,n_rows,n_zones,first_day,last_day,zero_trip_days
0,7688,248,2024-03-01,2024-03-31,2250


,pickup_zone_id,n_days
0,1,31
1,2,31
2,3,31
3,4,31
4,6,31
5,7,31
6,8,31
7,9,31
8,10,31
9,11,31


## 4. Time-series ML features
`ROWS` рахує рядки, `RANGE` — календарний інтервал. Для 7 і 30 днів включно з поточним потрібні відповідно 6 і 29 попередніх днів. `AVG` та `STDDEV_SAMP` ігнорують невідомі ціни; окремо показую кількість відомих значень. Поточний день входить у фічі, тому вони доступні після його завершення — для прогнозу наступного дня, а не поточного. За один місяць повна 30-денна lag-історія є лише 31 березня.

### Lag 1/7/30

In [4]:
run_sql("""
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
""")

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-01,1,NaN,NaN,None
1,1,2024-03-02,2,1.0,NaN,None
2,1,2024-03-03,0,2.0,NaN,None
3,1,2024-03-04,0,0.0,NaN,None
4,1,2024-03-05,0,0.0,NaN,None
5,1,2024-03-06,0,0.0,NaN,None
6,1,2024-03-07,0,0.0,NaN,None
7,1,2024-03-08,1,0.0,1.0,None
8,1,2024-03-09,0,1.0,2.0,None
9,1,2024-03-10,0,0.0,0.0,None


### Rolling 7-day average

In [5]:
run_sql("""
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window,
    COUNT(avg_fare) OVER w7 AS observed_fare_days
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
""")

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window,observed_fare_days
0,1,2024-03-01,105.00,105.0000000000000000,1,1
1,1,2024-03-02,95.00,100.0000000000000000,2,2
2,1,2024-03-03,None,100.0000000000000000,3,2
3,1,2024-03-04,None,100.0000000000000000,4,2
4,1,2024-03-05,None,100.0000000000000000,5,2
5,1,2024-03-06,None,100.0000000000000000,6,2
6,1,2024-03-07,None,100.0000000000000000,7,2
7,1,2024-03-08,105.00,100.0000000000000000,7,2
8,1,2024-03-09,None,105.0000000000000000,7,1
9,1,2024-03-10,None,105.0000000000000000,7,1


### Rolling 30-day volatility

In [6]:
run_sql("""
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window,
    COUNT(avg_fare) OVER w30 AS observed_fare_days
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
""")

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window,observed_fare_days
0,1,2024-03-01,105.00,None,1,1
1,1,2024-03-02,95.00,7.0710678118654752,2,2
2,1,2024-03-03,None,7.0710678118654752,3,2
3,1,2024-03-04,None,7.0710678118654752,4,2
4,1,2024-03-05,None,7.0710678118654752,5,2
5,1,2024-03-06,None,7.0710678118654752,6,2
6,1,2024-03-07,None,7.0710678118654752,7,2
7,1,2024-03-08,105.00,5.7735026918962576,8,3
8,1,2024-03-09,None,5.7735026918962576,9,3
9,1,2024-03-10,None,5.7735026918962576,10,3


### Expanding features

In [7]:
run_sql("""
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
""")

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-03-01,1,1,1.00000000000000000000
1,1,2024-03-02,2,3,1.5000000000000000
2,1,2024-03-03,0,3,1.00000000000000000000
3,1,2024-03-04,0,3,0.75000000000000000000
4,1,2024-03-05,0,3,0.60000000000000000000
5,1,2024-03-06,0,3,0.50000000000000000000
6,1,2024-03-07,0,3,0.42857142857142857143
7,1,2024-03-08,1,4,0.50000000000000000000
8,1,2024-03-09,0,4,0.44444444444444444444
9,1,2024-03-10,0,4,0.40000000000000000000


### Combined feature view

In [8]:
run_sql("""
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;

SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
""")

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-31,0,None,0.00,0,1,1,None,36.1494563506070509,12,0.38709677419354838710
1,2,2024-03-31,0,None,0.00,0,0,0,None,None,1,0.03225806451612903226
2,3,2024-03-31,0,None,0.00,0,1,1,51.7500000000000000,13.5255227194798218,18,0.58064516129032258065
3,4,2024-03-31,48,16.94,129.68,77,64,27,18.2085714285714286,2.1615656575625971,1004,32.3870967741935484
4,6,2024-03-31,0,None,0.00,0,0,0,85.6000000000000000,49.4154095399400692,3,0.09677419354838709677
5,7,2024-03-31,18,22.59,89.95,15,14,5,22.4657142857142857,5.0493117887718010,298,9.6129032258064516
6,8,2024-03-31,0,None,0.00,0,0,0,None,None,1,0.03225806451612903226
7,9,2024-03-31,0,None,0.00,1,1,0,14.2550000000000000,20.3969968611161225,7,0.22580645161290322581
8,10,2024-03-31,5,75.85,67.30,2,5,12,58.2571428571428571,12.3229412446680208,162,5.2258064516129032
9,11,2024-03-31,0,None,0.00,0,0,0,37.0000000000000000,11.7386777601037995,15,0.48387096774193548387


In [9]:
with engine.connect() as conn:
    assert conn.execute(text('SELECT COUNT(*) FROM t7_hw_taxi_daily')).scalar_one() > 100
    assert conn.execute(text('SELECT COUNT(*) FROM (SELECT pickup_zone_id FROM t7_hw_taxi_daily GROUP BY 1 HAVING COUNT(*) <> 31) s')).scalar_one() == 0
    assert conn.execute(text("SELECT COUNT(*) FROM v_t7_hw_taxi_features WHERE ts_day <> DATE '2024-03-31'")).scalar_one() == 0
    assert conn.execute(text('SELECT COUNT(*) FROM v_t7_hw_taxi_features')).scalar_one() > 0
    assert conn.execute(text("SELECT COUNT(*) FROM v_t7_hw_taxi_features f JOIN t7_hw_taxi_daily d ON d.pickup_zone_id=f.pickup_zone_id AND d.ts_day=f.ts_day-30 WHERE f.trips_lag_30d IS DISTINCT FROM d.trip_count")).scalar_one() == 0
print('Daily grid and calendar lag checks: OK')

Daily grid and calendar lag checks: OK


## 5. JSONB event-log
Я використала відтворюваний seed із завдання: 50 000 подій, п’ять типів і фіксовані дати. Це навчальні синтетичні дані, а не реальна поведінка користувачів.

In [10]:
run_sql("""
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;

SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;
""")

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


### Containment @>

In [11]:
run_sql("""
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;
""")

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'item..."
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'item..."
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'item..."
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'item..."
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'item..."


### Key existence ? / ?|

In [12]:
run_sql("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;

SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;
""")

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quanti..."
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3..."
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quanti..."


,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


### JSONPath

In [13]:
run_sql("""
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;

SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10;
""")

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


,event_id,order_id,total
0,253,o_100253,303.00
1,258,o_100258,308.00
2,263,o_100263,313.00
3,268,o_100268,318.00
4,273,o_100273,323.00
5,278,o_100278,328.00
6,283,o_100283,333.00
7,288,o_100288,338.00
8,293,o_100293,343.00
9,298,o_100298,348.00


## 6. Індекси та EXPLAIN (ANALYZE, BUFFERS)
Плани друкую повністю, без обрізання таблицею pandas. Не вимикаю `Seq Scan`: вибір плану залишаю PostgreSQL. Часи залежать від кешу й середовища; одиничні вимірювання не є стабільним benchmark. `jsonb_path_ops` підходить для containment; для індексації `?` / `?|` потрібен `jsonb_ops`. [Документація PostgreSQL](https://www.postgresql.org/docs/current/datatype-json.html#JSON-INDEXING).

In [14]:
run_sql("""
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;
""")

In [15]:
q_containment = "SELECT COUNT(*) FROM t7_hw_events WHERE payload @> jsonb_build_object('payment_method', 'card')"
q_device = "SELECT COUNT(*) FROM t7_hw_events WHERE payload ->> 'device' = 'mobile'"
explain('containment_before', q_containment)

containment_before
Aggregate  (cost=1758.23..1758.24 rows=1 width=8) (actual time=172.488..172.489 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=3294 width=0) (actual time=0.020..171.853 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=7
Planning Time: 0.297 ms
Execution Time: 172.522 ms


In [16]:
run_sql("""
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;

SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops'))
       AS gin_index_size;
""")

,gin_index_size
0,1144 kB


In [17]:
explain('containment_after', q_containment)
explain('referrer_after', "SELECT COUNT(*) FROM t7_hw_events WHERE payload @> jsonb_build_object('referrer', 'google')")

containment_after
Aggregate  (cost=1084.31..1084.32 rows=1 width=8) (actual time=15.849..15.851 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=32.50..1076.91 rows=2961 width=0) (actual time=1.156..15.309 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..31.76 rows=2961 width=0) (actual time=0.854..0.854 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=27
Planning Time: 0.591 ms
Execution Time: 17.524 ms
referrer_after
Aggregate  (cost=1096.08..1096.09 rows=1 width=8) (actual time=9.470..9.472 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=19.69..1092.81 rows=1308 width=0) (actual time=0.804..9

In [18]:
explain('device_before', q_device)

device_before
Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=88.090..88.091 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.033..86.678 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.087 ms
Execution Time: 88.127 ms


In [19]:
run_sql("""
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;
""")

In [20]:
explain('device_after', q_device)

device_after
Aggregate  (cost=1241.03..1241.04 rows=1 width=8) (actual time=21.401..21.403 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=96.36..1220.36 rows=8267 width=0) (actual time=1.596..19.871 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.29 rows=8267 width=0) (actual time=1.091..1.092 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=19 read=1
Planning Time: 0.496 ms
Execution Time: 21.524 ms


## 7. JSONB vs normalized checkout
Порівнюю однакову агрегацію на тих самих checkout-подіях. Нормалізована таблиця містить лише checkout, а JSONB-таблиця — усі типи подій: різниця часу включає і цей архітектурний ефект, а не лише вартість читання JSON.

In [21]:
run_sql("""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;

SELECT COUNT(*) AS n_checkouts
FROM t7_hw_checkouts_norm;
""")

,n_checkouts
0,10000


In [22]:
q_jsonb = "SELECT\n    payload ->> 'payment_method' AS payment_method,\n    AVG((payload ->> 'total')::NUMERIC) AS avg_total,\n    COUNT(*) AS n\nFROM t7_hw_events\nWHERE event_type = 'checkout'\nGROUP BY payload ->> 'payment_method'\nORDER BY avg_total DESC\nLIMIT 3"
q_normalized = 'SELECT\n    payment_method,\n    AVG(total) AS avg_total,\n    COUNT(*) AS n\nFROM t7_hw_checkouts_norm\nGROUP BY payment_method\nORDER BY avg_total DESC\nLIMIT 3'
jsonb_result = pd.read_sql(text(q_jsonb), engine)
normalized_result = pd.read_sql(text(q_normalized), engine)
display(jsonb_result)
display(normalized_result)
pd.testing.assert_frame_equal(jsonb_result.sort_values('payment_method').reset_index(drop=True),
                              normalized_result.sort_values('payment_method').reset_index(drop=True))
print('JSONB and normalized results match: OK')
explain('jsonb_aggregation', q_jsonb)
explain('normalized_aggregation', q_normalized)

,payment_method,avg_total,n
0,paypal,300.524752,3333
1,card,300.500000,3334
2,crypto,300.475248,3333


,payment_method,avg_total,n
0,paypal,300.524752,3333
1,card,300.500000,3334
2,crypto,300.475248,3333


JSONB and normalized results match: OK
jsonb_aggregation
Limit  (cost=1954.56..1954.56 rows=3 width=72) (actual time=35.689..35.691 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1954.56..1968.41 rows=5541 width=72) (actual time=35.687..35.688 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1799.82..1882.94 rows=5541 width=72) (actual time=35.618..35.658 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1649.97 rows=9990 width=132) (actual time=0.018..18.993 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning Time: 0.182 ms
Executi

In [23]:
assert len(plans) == 7
assert all('Execution Time:' in plan and 'Buffers:' in plan for plan in plans.values())
print('All seven EXPLAIN outputs saved in notebook.')

All seven EXPLAIN outputs saved in notebook.


## 8. Reflection

У цій роботі я підготувала часові ознаки з реальних поїздок NYC Taxi та порівняла способи зберігання подій. Для мене важливо було перевірити не лише SQL-синтаксис, а й зміст результатів. Тому я додала пропущені календарні дні: без цього lag міг би означати попередній запис замість потрібної дати. Невідому середню ціну я залишила NULL.

До створення GIN я побачила Seq Scan: запит перевіряв 50 000 подій і знаходив 3 334 оплати карткою. Total cost верхнього вузла становив 1756.57, actual time — 19.076..19.076 мс. Після індексації з’явилися Bitmap Index Scan і Bitmap Heap Scan, cost зменшився до 1060.31, actual time — до 2.199..2.199 мс. Повний Execution Time змінився з 19.089 до 2.210 мс. При цьому залишилося читання всіх 1 000 heap-блоків: відповідні події розподілені по таблиці. Отже, виграш не означає пропорційного зменшення звернень до сторінок.

GIN допоміг знайти JSONB-фрагмент через @>, але не прискорив текстовий вираз для device. Для нього я створила expression B-tree index. Час зменшився з 4.389 до 1.364 мс, а план також використав bitmap-сканування. За швидше читання доводиться платити місцем і підтримкою індексу під час INSERT та UPDATE. Вартість запису я окремо не вимірювала.

Для checkout нормалізована агрегація виконалася за 1.513 мс проти 5.706 мс у JSONB; результати збіглися. Обидва плани містили Seq Scan, але нормалізована таблиця читала лише checkout і займала менше сторінок. Я не вважаю ці одиничні вимірювання універсальним доказом: кеш, обсяг і середовище впливають на час, а cost не вимірюється в мілісекундах.

Я обрала б JSONB для мінливих атрибутів подій, а типізовані таблиці — для регулярної аналітики та ML-ознак. Новий refund у JSONB додала б через event_type і payload, оновивши валідацію та обробники. У нормалізованій моделі створила б таблицю повернень із типами, обмеженнями й посиланням на замовлення, а також міграцію та зміни ETL.